# 06 — Análisis de atípicos e IQR

Este notebook identifica extremos con el rango intercuartílico (IQR) y compara su tasa de churn con el resto de clientes. El objetivo no es borrar clientes automáticamente: primero verificamos si los extremos son ruido o información de negocio.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RAIZ = Path.cwd().resolve()
if RAIZ.name == 'notebooks':
    RAIZ = RAIZ.parent
train = pd.read_csv(RAIZ / 'data' / 'processed' / 'train_preparado.csv')
sns.set_theme(style='whitegrid', context='notebook')

VARIABLES = ['facturacion', 'mora', 'minutos', 'edad_anios', 'antiguedad_contrato_meses']
FACTOR_IQR = 1.5

## 1. Límites IQR y riesgo de los extremos

Un valor queda marcado como atípico si está por debajo de `Q1 − 1,5 × IQR` o por encima de `Q3 + 1,5 × IQR`. La comparación de churn responde la pregunta esencial: ¿los extremos parecen ruido o concentran riesgo?

Esta es una exploración descriptiva. En el modelo, los límites se aprenderán solo con cada fold de entrenamiento.

In [ ]:
filas = []
for variable in VARIABLES:
    q1, q3 = train[variable].quantile([.25, .75])
    iqr = q3 - q1
    limite_inferior = q1 - FACTOR_IQR * iqr
    limite_superior = q3 + FACTOR_IQR * iqr
    atipico = (train[variable] < limite_inferior) | (train[variable] > limite_superior)
    filas.append({
        'variable': variable,
        'límite inferior': limite_inferior,
        'límite superior': limite_superior,
        'clientes atípicos': atipico.sum(),
        '% atípicos': atipico.mean(),
        'churn atípicos': train.loc[atipico, 'resultado'].mean(),
        'churn no atípicos': train.loc[~atipico, 'resultado'].mean(),
    })

resumen_iqr = pd.DataFrame(filas)
display(resumen_iqr.style.format({'límite inferior': '{:,.1f}', 'límite superior': '{:,.1f}', '% atípicos': '{:.1%}', 'churn atípicos': '{:.1%}', 'churn no atípicos': '{:.1%}'}))

## 2. Visualizar los extremos por resultado

Las cajas de bigotes muestran la forma de cada variable en clientes con y sin churn. No eliminamos filas a partir de esta gráfica: sirve para entender dónde aparecen los extremos.

In [ ]:
fig, axes = plt.subplots(1, len(VARIABLES), figsize=(18, 4.5))
for ax, variable in zip(axes, VARIABLES):
    sns.boxplot(data=train, x='resultado', y=variable, hue='resultado', legend=False, showfliers=True, ax=ax)
    ax.set(title=variable.replace('_', ' '), xlabel='Resultado')
plt.tight_layout(); plt.show()

## 3. Decisión de modelado

Los extremos de mora, minutos y facturación tienen una tasa de churn mayor que los clientes no atípicos. Por eso no se eliminan clientes ni se recorta mora: mora es precisamente la variable que el spline está aprovechando.

El experimento final de hoy usa `spline_mora_3_nudos_iqr_numericas`: conserva la spline de mora y limita por IQR las otras variables numéricas antes de escalarlas. Así probamos robustez de distancias sin destruir la señal no lineal de mora.

In [ ]:
display(pd.DataFrame({
    'acción': ['No eliminar filas', 'No recortar mora', 'Recortar otras numéricas dentro de CV'],
    'razón': [
        'Test debe conservar todos sus clientes y los extremos pueden ser casos reales.',
        'Mora extrema concentra churn y su forma no lineal ya mejoró los modelos.',
        'Facturación, minutos, edad y antigüedad se vuelven menos sensibles a extremos al medir distancias.'
    ]
}))